In [ ]:
# SOLUTION: use PyTorch instead of TensorFlow
# PyTorch is compatible with Python 3.13

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
np.random.seed(42)

print("Imports successful.")
print(f"pandas version: {pd.__version__}")
print(f"numpy version: {np.__version__}")

Imports successful.
pandas version: 2.2.2
numpy version: 2.0.2


In [ ]:
# Install PyTorch (compatible with Python 3.13)
%pip install torch torchvision torchaudio --quiet

In [ ]:
# Import PyTorch
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# Set random seeds
torch.manual_seed(42)

print("PyTorch imported successfully.")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"MPS (Apple Silicon) available: {torch.backends.mps.is_available() if hasattr(torch.backends, 'mps') else False}")

PyTorch imported successfully.
PyTorch version: 2.9.0+cu126
CUDA available: False
MPS (Apple Silicon) available: False


## 1. Import Libraries and Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
np.random.seed(42)

# Note: TensorFlow imports removed to avoid kernel incompatibility on Python 3.13.
print("Environment ready. TensorFlow imports removed to ensure compatibility with this Python version.")

Environment ready. TensorFlow imports removed to ensure compatibility with this Python version.


In [ ]:
# Load the dataset
df = pd.read_csv('../data/raw/car_prices.csv')

print("Dataset shape:", df.shape)
print("\nFirst few rows:")
df.head()

Dataset shape: (558837, 16)

First few rows:


,year,make,model,trim,body,transmission,vin,state,condition,odometer,color,interior,seller,mmr,sellingprice,saledate
0,2015,Kia,Sorento,LX,SUV,automatic,5xyktca69fg566472,ca,5.0,16639.0,white,black,kia motors america inc,20500.0,21500.0,Tue Dec 16 2014 12:30:00 GMT-0800 (PST)
1,2015,Kia,Sorento,LX,SUV,automatic,5xyktca69fg561319,ca,5.0,9393.0,white,beige,kia motors america inc,20800.0,21500.0,Tue Dec 16 2014 12:30:00 GMT-0800 (PST)
2,2014,BMW,3 Series,328i SULEV,Sedan,automatic,wba3c1c51ek116351,ca,45.0,1331.0,gray,black,financial services remarketing (lease),31900.0,30000.0,Thu Jan 15 2015 04:30:00 GMT-0800 (PST)
3,2015,Volvo,S60,T5,Sedan,automatic,yv1612tb4f1310987,ca,41.0,14282.0,white,black,volvo na rep/world omni,27500.0,27750.0,Thu Jan 29 2015 04:30:00 GMT-0800 (PST)
4,2014,BMW,6 Series Gran Coupe,650i,Sedan,automatic,wba6b2c57ed129731,ca,43.0,2641.0,gray,black,financial services remarketing (lease),66000.0,67000.0,Thu Dec 18 2014 12:30:00 GMT-0800 (PST)


In [ ]:
# Data overview
print("Dataset Info:")
print(df.info())
print("\n" + "="*50)
print("\nBasic Statistics:")
print(df.describe())
print("\n" + "="*50)
print("\nMissing Values:")
print(df.isnull().sum())
print("\n" + "="*50)
print("\nColumn Names:")
print(df.columns.tolist())

Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 558837 entries, 0 to 558836
Data columns (total 16 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   year          558837 non-null  int64  
 1   make          548536 non-null  object 
 2   model         548438 non-null  object 
 3   trim          548186 non-null  object 
 4   body          545642 non-null  object 
 5   transmission  493485 non-null  object 
 6   vin           558833 non-null  object 
 7   state         558837 non-null  object 
 8   condition     547017 non-null  float64
 9   odometer      558743 non-null  float64
 10  color         558088 non-null  object 
 11  interior      558088 non-null  object 
 12  seller        558837 non-null  object 
 13  mmr           558799 non-null  float64
 14  sellingprice  558825 non-null  float64
 15  saledate      558825 non-null  object 
dtypes: float64(4), int64(1), object(11)
memory usage: 68.2+ MB
None


Basic Statis

## 2. Data Preparation for MLP + Embedding

In [ ]:
# Handle missing values
print("Handling missing values...")
df_clean = df.copy()

# For numerical columns: fill with median
numerical_cols = df_clean.select_dtypes(include=[np.number]).columns
for col in numerical_cols:
    if df_clean[col].isnull().sum() > 0:
        df_clean[col].fillna(df_clean[col].median(), inplace=True)

# For categorical columns: fill with mode
categorical_cols = df_clean.select_dtypes(include=['object']).columns
for col in categorical_cols:
    if df_clean[col].isnull().sum() > 0:
        df_clean[col].fillna(df_clean[col].mode()[0], inplace=True)

print(f"Missing values after cleaning: {df_clean.isnull().sum().sum()}")
print(f"\nDataset shape: {df_clean.shape}")

Handling missing values...
Missing values after cleaning: 0

Dataset shape: (558837, 16)


In [ ]:
# Identify target and features
# Assuming 'price' or 'sellingprice' or similar is the target
target_candidates = ['price', 'sellingprice', 'selling_price', 'Price', 'SellingPrice']
target_col = None

for candidate in target_candidates:
    if candidate in df_clean.columns:
        target_col = candidate
        break

if target_col is None:
    # If not found, assume last column or first numeric column
    numeric_cols = df_clean.select_dtypes(include=[np.number]).columns
    target_col = numeric_cols[-1] if len(numeric_cols) > 0 else df_clean.columns[-1]

print(f"Target column identified: {target_col}")
print(f"Target statistics:\n{df_clean[target_col].describe()}")

# Separate features and target
y = df_clean[target_col].values
X = df_clean.drop(columns=[target_col])

print(f"\nFeatures shape: {X.shape}")
print(f"Target shape: {y.shape}")

Target column identified: sellingprice
Target statistics:
count    558837.000000
mean      13611.326356
std        9749.399466
min           1.000000
25%        6900.000000
50%       12100.000000
75%       18200.000000
max      230000.000000
Name: sellingprice, dtype: float64

Features shape: (558837, 15)
Target shape: (558837,)


In [ ]:
# Identify categorical and numerical features
categorical_features = X.select_dtypes(include=['object']).columns.tolist()
numerical_features = X.select_dtypes(include=[np.number]).columns.tolist()

print(f"Categorical features ({len(categorical_features)}): {categorical_features}")
print(f"\nNumerical features ({len(numerical_features)}): {numerical_features}")

# Encode categorical features for embedding
label_encoders = {}
embedding_dims = {}
categorical_encoded = {}

for col in categorical_features:
    le = LabelEncoder()
    categorical_encoded[col] = le.fit_transform(X[col].astype(str))
    label_encoders[col] = le

    # Calculate embedding dimension (rule of thumb: min(50, (cardinality+1)//2))
    n_categories = len(le.classes_)
    embedding_dim = min(50, (n_categories + 1) // 2)
    embedding_dims[col] = (n_categories, embedding_dim)

    print(f"{col}: {n_categories} categories -> embedding dim = {embedding_dim}")

print(f"\nTotal categorical features with embeddings: {len(categorical_features)}")

Categorical features (11): ['make', 'model', 'trim', 'body', 'transmission', 'vin', 'state', 'color', 'interior', 'seller', 'saledate']

Numerical features (4): ['year', 'condition', 'odometer', 'mmr']
make: 96 categories -> embedding dim = 48
model: 973 categories -> embedding dim = 50
trim: 1963 categories -> embedding dim = 50
body: 87 categories -> embedding dim = 44
transmission: 4 categories -> embedding dim = 2
vin: 550297 categories -> embedding dim = 50
state: 64 categories -> embedding dim = 32
color: 46 categories -> embedding dim = 23
interior: 17 categories -> embedding dim = 9
seller: 14263 categories -> embedding dim = 50
saledate: 3766 categories -> embedding dim = 50

Total categorical features with embeddings: 11


In [ ]:
# Prepare numerical features (standardize)
scaler = StandardScaler()
numerical_data = scaler.fit_transform(X[numerical_features]) if len(numerical_features) > 0 else np.array([]).reshape(len(X), 0)

print(f"Numerical data shape: {numerical_data.shape}")

# Split data into train and test sets
test_size = 0.2
random_state = 42

# Split numerical data
X_num_train, X_num_test, y_train, y_test = train_test_split(
    numerical_data, y, test_size=test_size, random_state=random_state
)

# Split categorical data
X_cat_train = {}
X_cat_test = {}

for col in categorical_features:
    cat_train, cat_test = train_test_split(
        categorical_encoded[col], test_size=test_size, random_state=random_state
    )
    X_cat_train[col] = cat_train
    X_cat_test[col] = cat_test

print(f"\nTrain set size: {len(X_num_train)}")
print(f"Test set size: {len(X_num_test)}")
print(f"Target mean (train): {y_train.mean():.2f}")
print(f"Target std (train): {y_train.std():.2f}")

Numerical data shape: (558837, 4)

Train set size: 447069
Test set size: 111768
Target mean (train): 13615.00
Target std (train): 9766.68


## 3. Build MLP + Embedding Model

In [ ]:
# PyTorch Model Definition with Embeddings
class MLPEmbeddingModel(nn.Module):
    def __init__(self, embedding_dims, numerical_input_dim):
        """
        MLP model with embedding layers for categorical features

        Args:
            embedding_dims: dict of (n_categories, embedding_dim) for each categorical feature
            numerical_input_dim: dimension of numerical features
        """
        super(MLPEmbeddingModel, self).__init__()

        # Embedding layers for categorical features
        self.embeddings = nn.ModuleDict()
        total_embedding_dim = 0

        for col, (n_categories, embedding_dim) in embedding_dims.items():
            self.embeddings[col] = nn.Embedding(n_categories, embedding_dim)
            total_embedding_dim += embedding_dim

        # Calculate total input dimension
        total_input_dim = total_embedding_dim + numerical_input_dim

        # MLP layers
        self.fc1 = nn.Linear(total_input_dim, 256)
        self.bn1 = nn.BatchNorm1d(256)
        self.dropout1 = nn.Dropout(0.3)

        self.fc2 = nn.Linear(256, 128)
        self.bn2 = nn.BatchNorm1d(128)
        self.dropout2 = nn.Dropout(0.3)

        self.fc3 = nn.Linear(128, 64)
        self.bn3 = nn.BatchNorm1d(64)
        self.dropout3 = nn.Dropout(0.2)

        self.fc4 = nn.Linear(64, 32)
        self.dropout4 = nn.Dropout(0.2)

        self.output = nn.Linear(32, 1)

        self.relu = nn.ReLU()

    def forward(self, categorical_inputs, numerical_input):
        """
        Forward pass

        Args:
            categorical_inputs: dict of tensors for each categorical feature
            numerical_input: tensor of numerical features
        """
        # Process embeddings
        embedding_outputs = []
        for col, embedding_layer in self.embeddings.items():
            emb = embedding_layer(categorical_inputs[col])
            embedding_outputs.append(emb)

        # Concatenate all features
        if len(embedding_outputs) > 0:
            x = torch.cat(embedding_outputs + [numerical_input], dim=1)
        else:
            x = numerical_input

        # MLP layers
        x = self.fc1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.dropout1(x)

        x = self.fc2(x)
        x = self.bn2(x)
        x = self.relu(x)
        x = self.dropout2(x)

        x = self.fc3(x)
        x = self.bn3(x)
        x = self.relu(x)
        x = self.dropout3(x)

        x = self.fc4(x)
        x = self.relu(x)
        x = self.dropout4(x)

        x = self.output(x)

        return x

# Build the model
device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model = MLPEmbeddingModel(embedding_dims, len(numerical_features))
model = model.to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Model created successfully.")
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"\nModel Architecture:")
print(model)

Using device: cpu
Model created successfully.
Total parameters: 28,724,692
Trainable parameters: 28,724,692

Model Architecture:
MLPEmbeddingModel(
  (embeddings): ModuleDict(
    (make): Embedding(96, 48)
    (model): Embedding(973, 50)
    (trim): Embedding(1963, 50)
    (body): Embedding(87, 44)
    (transmission): Embedding(4, 2)
    (vin): Embedding(550297, 50)
    (state): Embedding(64, 32)
    (color): Embedding(46, 23)
    (interior): Embedding(17, 9)
    (seller): Embedding(14263, 50)
    (saledate): Embedding(3766, 50)
  )
  (fc1): Linear(in_features=412, out_features=256, bias=True)
  (bn1): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (dropout1): Dropout(p=0.3, inplace=False)
  (fc2): Linear(in_features=256, out_features=128, bias=True)
  (bn2): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (dropout2): Dropout(p=0.3, inplace=False)
  (fc3): Linear(in_features=128, out_features=64, bias=True)
  (bn3):

## 4. Train the Model

In [ ]:
# PyTorch Dataset class
class CarPriceDataset(Dataset):
    def __init__(self, categorical_data, numerical_data, targets):
        self.categorical_data = categorical_data
        self.numerical_data = torch.FloatTensor(numerical_data)
        self.targets = torch.FloatTensor(targets).reshape(-1, 1)

    def __len__(self):
        return len(self.targets)

    def __getitem__(self, idx):
        cat_dict = {col: torch.LongTensor([self.categorical_data[col][idx]])
                    for col in self.categorical_data.keys()}
        return cat_dict, self.numerical_data[idx], self.targets[idx]

# Clean NaN and infinite values before creating datasets
X_num_train = np.nan_to_num(
    np.asarray(X_num_train, dtype=np.float32),
    nan=0.0,
    posinf=0.0,
    neginf=0.0
)
X_num_test = np.nan_to_num(
    np.asarray(X_num_test, dtype=np.float32),
    nan=0.0,
    posinf=0.0,
    neginf=0.0
)

y_train = np.nan_to_num(
    np.asarray(y_train, dtype=np.float32),
    nan=0.0,
    posinf=0.0,
    neginf=0.0
)
y_test = np.nan_to_num(
    np.asarray(y_test, dtype=np.float32),
    nan=0.0,
    posinf=0.0,
    neginf=0.0
)

print("Remaining NaN values:")
print("X_num_train:", np.isnan(X_num_train).sum())
print("X_num_test:", np.isnan(X_num_test).sum())
print("y_train:", np.isnan(y_train).sum())
print("y_test:", np.isnan(y_test).sum())

# Create datasets
train_dataset = CarPriceDataset(X_cat_train, X_num_train, y_train)
test_dataset = CarPriceDataset(X_cat_test, X_num_test, y_test)

# Create dataloaders
train_loader = DataLoader(train_dataset, batch_size=512, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=512, shuffle=False)

print("Datasets created.")
print(f"Train batches: {len(train_loader)}")
print(f"Test batches: {len(test_loader)}")

# Training setup
criterion = nn.HuberLoss(delta=1000.0)
optimizer = optim.Adam(model.parameters(), lr=0.0001)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

# Training function
def train_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    for cat_inputs, num_inputs, targets in dataloader:
        # Move to device
        cat_inputs = {k: v.squeeze().to(device) for k, v in cat_inputs.items()}
        num_inputs = num_inputs.to(device)
        targets = targets.to(device)

        # Forward pass
        optimizer.zero_grad()
        outputs = model(cat_inputs, num_inputs)
        loss = criterion(outputs, targets)

        # Backward pass
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)

# Validation function
def validate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0
    all_preds = []
    all_targets = []

    with torch.no_grad():
        for cat_inputs, num_inputs, targets in dataloader:
            cat_inputs = {k: v.squeeze().to(device) for k, v in cat_inputs.items()}
            num_inputs = num_inputs.to(device)
            targets = targets.to(device)

            outputs = model(cat_inputs, num_inputs)
            loss = criterion(outputs, targets)

            total_loss += loss.item()
            all_preds.extend(outputs.cpu().numpy())
            all_targets.extend(targets.cpu().numpy())

    return total_loss / len(dataloader), np.array(all_preds).flatten(), np.array(all_targets).flatten()

# Training loop
print("\n" + "="*60)
print("TRAINING MODEL")
print("="*60)

num_epochs = 20
best_val_loss = float('inf')
patience = 5
patience_counter = 0
history = {'train_loss': [], 'val_loss': [], 'val_mae': []}

for epoch in range(num_epochs):
    train_loss = train_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_preds, val_targets = validate(model, test_loader, criterion, device)
    val_mae = mean_absolute_error(val_targets, val_preds)

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_mae'].append(val_mae)

    scheduler.step(val_loss)

    if (epoch + 1) % 10 == 0:
        print(f"Epoch [{epoch+1}/{num_epochs}] - Train Loss: {train_loss:.2f}, Val Loss: {val_loss:.2f}, Val MAE: {val_mae:.2f}")

    # Early stopping
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        best_model_state = model.state_dict().copy()
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"Early stopping at epoch {epoch+1}")
            model.load_state_dict(best_model_state)
            break

print("\nTraining completed!")
print(f"Best validation loss: {best_val_loss:.2f}")

Datasets created.
Train batches: 13971
Test batches: 3493

TRAINING MODEL


In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Loss
epochs_range = range(1, len(history['train_loss']) + 1)
axes[0].plot(epochs_range, history['train_loss'], label='Training Loss', marker='o', markersize=3)
axes[0].plot(epochs_range, history['val_loss'], label='Validation Loss', marker='o', markersize=3)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss (MSE)')
axes[0].set_title('Model Loss During Training')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# MAE
axes[1].plot(epochs_range, history['val_mae'], label='Validation MAE', marker='o', markersize=3, color='orange')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('MAE')
axes[1].set_title('Mean Absolute Error During Training')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Best validation loss: {min(history['val_loss']):.2f}")
print(f"Best validation MAE: {min(history['val_mae']):.2f}")

## 5. Test and Evaluate Results

In [ ]:
# Make predictions on test set
model.eval()
y_pred = []

with torch.no_grad():
    for cat_inputs, num_inputs, _ in test_loader:
        cat_inputs = {k: v.squeeze().to(device) for k, v in cat_inputs.items()}
        num_inputs = num_inputs.to(device)

        outputs = model(cat_inputs, num_inputs)
        y_pred.extend(outputs.cpu().numpy())

y_pred = np.array(y_pred).flatten()

# Calculate metrics
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
mape = np.mean(np.abs((y_test - y_pred) / (y_test + 1e-10))) * 100

print("="*60)
print("MODEL PERFORMANCE ON TEST SET")
print("="*60)
print(f"Mean Squared Error (MSE):       {mse:,.2f}")
print(f"Root Mean Squared Error (RMSE):  {rmse:,.2f}")
print(f"Mean Absolute Error (MAE):       {mae:,.2f}")
print(f"R² Score:                        {r2:.4f}")
print(f"Mean Absolute Percentage Error:  {mape:.2f}%")
print("="*60)

# Additional statistics
print(f"\nActual Price Range:     ${y_test.min():,.2f} - ${y_test.max():,.2f}")
print(f"Predicted Price Range:  ${y_pred.min():,.2f} - ${y_pred.max():,.2f}")
print(f"Mean Actual Price:      ${y_test.mean():,.2f}")
print(f"Mean Predicted Price:   ${y_pred.mean():,.2f}")

In [ ]:
# Visualization of results
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Actual vs Predicted scatter plot
axes[0, 0].scatter(y_test, y_pred, alpha=0.5, s=20)
axes[0, 0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
axes[0, 0].set_xlabel('Actual Price')
axes[0, 0].set_ylabel('Predicted Price')
axes[0, 0].set_title(f'Actual vs Predicted Prices (R² = {r2:.4f})')
axes[0, 0].grid(True, alpha=0.3)

# 2. Residual plot
residuals = y_test - y_pred
axes[0, 1].scatter(y_pred, residuals, alpha=0.5, s=20)
axes[0, 1].axhline(y=0, color='r', linestyle='--', lw=2)
axes[0, 1].set_xlabel('Predicted Price')
axes[0, 1].set_ylabel('Residuals')
axes[0, 1].set_title('Residual Plot')
axes[0, 1].grid(True, alpha=0.3)

# 3. Distribution of residuals
axes[1, 0].hist(residuals, bins=50, edgecolor='black', alpha=0.7)
axes[1, 0].axvline(x=0, color='r', linestyle='--', lw=2)
axes[1, 0].set_xlabel('Residuals')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].set_title(f'Distribution of Residuals (Mean: {residuals.mean():.2f})')
axes[1, 0].grid(True, alpha=0.3)

# 4. Prediction error distribution
errors = np.abs(y_test - y_pred)
axes[1, 1].hist(errors, bins=50, edgecolor='black', alpha=0.7, color='orange')
axes[1, 1].axvline(x=mae, color='r', linestyle='--', lw=2, label=f'MAE = {mae:.2f}')
axes[1, 1].set_xlabel('Absolute Error')
axes[1, 1].set_ylabel('Frequency')
axes[1, 1].set_title('Distribution of Absolute Errors')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Sample predictions comparison
print("\n" + "="*80)
print("SAMPLE PREDICTIONS (First 20 Test Samples)")
print("="*80)
comparison_df = pd.DataFrame({
    'Actual Price': y_test[:20],
    'Predicted Price': y_pred[:20],
    'Difference': y_test[:20] - y_pred[:20],
    'Error %': np.abs((y_test[:20] - y_pred[:20]) / y_test[:20] * 100)
})
print(comparison_df.to_string(index=False))
print("="*80)

## 6. Insights and Analysis

In [ ]:
# Analyze embedding representations
print("="*80)
print("EMBEDDING ANALYSIS")
print("="*80)

for col in categorical_features:
    n_categories, embedding_dim = embedding_dims[col]
    print(f"\n{col}:")
    print(f"  - Number of unique values: {n_categories}")
    print(f"  - Embedding dimension: {embedding_dim}")
    print(f"  - Total parameters: {n_categories * embedding_dim:,}")

# Calculate total parameters
embedding_params = sum([n_cat * emb_dim for n_cat, emb_dim in embedding_dims.values()])

print(f"\n{'='*80}")
print(f"Total model parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Embedding layer parameters: {embedding_params:,} ({embedding_params/total_params*100:.2f}%)")
print(f"MLP layer parameters: {total_params - embedding_params:,} ({(total_params-embedding_params)/total_params*100:.2f}%)")
print("="*80)

In [ ]:
# Error analysis by price range
print("\n" + "="*80)
print("ERROR ANALYSIS BY PRICE RANGE")
print("="*80)

# Define price ranges
percentiles = [0, 25, 50, 75, 100]
price_ranges = np.percentile(y_test, percentiles)

for i in range(len(percentiles) - 1):
    mask = (y_test >= price_ranges[i]) & (y_test < price_ranges[i+1])
    if i == len(percentiles) - 2:  # Last range should be inclusive
        mask = (y_test >= price_ranges[i]) & (y_test <= price_ranges[i+1])

    range_actual = y_test[mask]
    range_pred = y_pred[mask]

    if len(range_actual) > 0:
        range_mae = mean_absolute_error(range_actual, range_pred)
        range_r2 = r2_score(range_actual, range_pred)
        range_mape = np.mean(np.abs((range_actual - range_pred) / range_actual)) * 100

        print(f"\nPrice Range: ${price_ranges[i]:,.0f} - ${price_ranges[i+1]:,.0f}")
        print(f"  Number of samples: {len(range_actual)}")
        print(f"  MAE: ${range_mae:,.2f}")
        print(f"  R² Score: {range_r2:.4f}")
        print(f"  MAPE: {range_mape:.2f}%")

print("="*80)

In [ ]:
# Final comprehensive insights
print("\n" + "="*80)
print("COMPREHENSIVE INSIGHTS - MLP + EMBEDDING WITH PYTORCH")
print("="*80)

insights = f"""
1) SUMMARY:
   - Switched from TensorFlow to PyTorch for compatibility with Python 3.13.
   - The model can run on MPS (Apple Silicon), CUDA (GPU), or CPU depending on availability.

2) MODEL ARCHITECTURE:
   - Categorical features are handled with Embedding layers.
   - Embeddings convert each category into dense vectors that capture relationships.
   - MLP architecture: 256 → 128 → 64 → 32 → 1 with BatchNorm and Dropout for regularization.

3) PERFORMANCE SUMMARY:
   - R² Score: {r2:.4f}
   - RMSE: ${rmse:,.2f}
   - MAE: ${mae:,.2f}
   - MAPE: {mape:.2f}%

4) MODEL METRICS & PARAMETERS:
   - Total parameters: {total_params:,}
   - Embedding parameters: {embedding_params:,} ({embedding_params/total_params*100:.1f}%)
   - Number of categorical features: {len(categorical_features)}
   - Number of numerical features: {len(numerical_features)}
   - Training samples: {len(X_num_train):,}
   - Test samples: {len(X_num_test):,}

5) RECOMMENDATIONS:
   - If R² is low or MAPE > 15%, consider collecting more data or adding domain features.
   - Extract embeddings to analyze relationships between categories or to use in other models.
   - Tune hyperparameters (learning rate, layer sizes, dropout) or try ensembling for better accuracy.

6) NOTES ON KERNEL CRASH:
   - Kernel crashes were caused by importing TensorFlow under Python 3.13.
   - Using PyTorch avoids that issue and provides stable execution on this environment.
"""

print(insights)
print("="*80)

## 7. Save Model (Optional)

In [ ]:
# Save the trained PyTorch model
model_path = 'car_price_mlp_embedding_model.pth'
torch.save(model.state_dict(), model_path)
print(f"Model state_dict saved to: {model_path}")

# Save preprocessors for future use
import pickle

preprocessors = {
    'label_encoders': label_encoders,
    'scaler': scaler,
    'embedding_dims': embedding_dims,
    'categorical_features': categorical_features,
    'numerical_features': numerical_features,
    'target_col': target_col
}

with open('preprocessors.pkl', 'wb') as f:
    pickle.dump(preprocessors, f)

print("Preprocessors saved to: preprocessors.pkl")
print("\nTo load and use the model later:")
print("  model = MLPEmbeddingModel(preprocessors['embedding_dims'], len(preprocessors['numerical_features']))")
print("  model.load_state_dict(torch.load('car_price_mlp_embedding_model.pth', map_location='cpu'))")
print("  model.eval()")